In [1]:
import numpy as np

def generate_continious_occlusion_data(screen_size=16,
                            num_steps=1000,
                            force_trial=False,
                            cutoff_percent=None):

    cue_width = 5
    screen_size = screen_size - 1

    base_steps_for_full_travel = 880
    pixels_per_step_slow = screen_size / base_steps_for_full_travel

    # continuous velocity
    v_min = pixels_per_step_slow * 0.7
    v_max = pixels_per_step_slow * 6.5
    velocity = np.random.uniform(v_min, v_max)
    velocity_norm = (velocity - v_min) / (v_max - v_min)

    # cutoff
    if cutoff_percent is None:
        cutoff_percent = np.random.choice([0.33, 0.66])

    predicted_time = int(np.ceil(screen_size / velocity))
    predicted_time = min(predicted_time, num_steps - 1)

    occlusion_distance = screen_size * cutoff_percent
    steps_visible = max(int(occlusion_distance / velocity), 10)

    sequence = []
    position = 0.0

    for t in range(num_steps):
        frame = np.ones(screen_size)
        if t < steps_visible:
            start = int(position)
            end = min(start + cue_width, screen_size)
            if start < screen_size:
                frame[start:end] = 5
            position += velocity

        sequence.append(frame)

    sequence = np.array(sequence)

    # add velocity channel
    v_chan = np.repeat([[velocity_norm]], num_steps, axis=0)
    sequence = np.concatenate((sequence, v_chan), axis=1)

    return sequence, predicted_time, velocity, cutoff_percent

In [2]:
def generate_occlusion_data(screen_size=16,
                            num_steps=1000,
                            seed=None, force_trial=False, ttype=None):

    # if seed is not None:
    #     np.random.seed(seed)

    speed_chooser = np.random.randint(0, 2)
    cutoff_chooser = np.random.randint(0, 2)

    speed_list = ['slow', 'fast']
    cutoff_list = [0.33, 0.66]
    if not force_trial:
        # Choose speed and occlusion point
        speed_type = speed_list[speed_chooser]
        cutoff_percent = cutoff_list[cutoff_chooser]
    else:
        if ttype == 'slow_33':
            speed_type = 'slow'
            cutoff_percent = 0.33
        elif ttype == 'slow_66':
            speed_type = 'slow'
            cutoff_percent = 0.66
        elif ttype == 'fast_33':
            speed_type = 'fast'
            cutoff_percent = 0.33
        elif ttype == 'fast_66':
            speed_type = 'fast'
            cutoff_percent = 0.66

    cue_width = 5
    screen_size = screen_size - 1
    # Design: we want slow motion to take ~960 steps to cross 64 pixels
    base_steps_for_full_travel = 880
    pixels_per_step_slow = screen_size / base_steps_for_full_travel  # ≈ 0.0667 px/step

    if speed_type == 'slow':
        velocity = pixels_per_step_slow                    # ~0.0667 px/step → takes 960 steps
        v_type = [0]
    else:
        velocity = pixels_per_step_slow * 6                # ~0.4 px/step → takes 160 steps
        v_type = [1]

    # Total predicted time to cross full screen
    predicted_time = int(np.ceil(screen_size / velocity))
    predicted_time = min(predicted_time, num_steps - 1)  # safety

    # How far does it travel before occlusion?
    occlusion_distance = screen_size * cutoff_percent
    steps_visible = int(occlusion_distance / velocity)
    steps_visible = max(steps_visible, 10)  # at least some motion visible

    # Generate sequence
    sequence = []
    position = 0.0  # sub-pixel precision for smooth slow motion

    for t in range(num_steps):
        frame = np.ones(screen_size)

        if t < steps_visible:
            start = int(position)
            end = min(start + cue_width, screen_size)
            if start < screen_size:
                frame[start:end] = 5  # bright block
            position += velocity
        # after occlusion: stays dark

        sequence.append(frame)
    sequence = np.array(sequence)
    # v_type = np.repeat(np.array(v_type).reshape(1, -1), num_steps, axis=0)
    # sequence = np.concatenate((sequence, v_type), axis=1) 
    return np.array(sequence), predicted_time, speed_type, cutoff_percent, steps_visible

if __name__ == "__main__":
    import matplotlib.pyplot as plt
    num_steps = 1000
    seq, pt, speed, cutoff, ot = generate_occlusion_data(seed=42, screen_size=19)
    print(seq.shape)
    print(f"Predicted time: {pt}, Speed: {speed}, Cutoff: {cutoff}")
    plt.rcParams.update({'font.size': 14})
    plt.rcParams['figure.dpi'] = 400
    plt.imshow(seq.T, aspect='auto', cmap='gray_r')
    plt.axvline(pt, color='red', linestyle='--', label='Objective Time')
    plt.axvline(ot, color='blue', linestyle='--', label='Occlusion Point')
    
    plt.title('Generated Occlusion Motion Prediction Sequence')
    plt.xticks(np.linspace(0, num_steps, 10), np.linspace(0, 20, 10, dtype=int))
    plt.yticks(np.arange(0, 19, 2))
    plt.xlabel('Time (s)')
    plt.ylabel('Screen Pixels')
    plt.legend()
    plt.savefig('occlusion_motion_prediction_sequence.png')
    plt.show()

In [ ]:
# @title
import torch
from torch import nn
from ffrnn.fftorch import FF
from IPython.display import clear_output

In [ ]:
def cReLU(x):
    return torch.complex(torch.relu(torch.real(x)),
                         torch.relu(torch.imag(x)))

def cLeakyReLU(x):
    return torch.complex(torch.nn.LeakyReLU()(torch.real(x)),
                         torch.nn.LeakyReLU()(torch.imag(x)))

class PosLinear(nn.Module):
    def __init__(self, in_features, out_features, bias=True):
        super(PosLinear, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        
        # We define the weight as a Parameter. 
        # Since we will use exp(), we initialize with small values 
        # so that exp(0) starts around 1.0.
        self.weight = nn.Parameter(torch.randn(out_features, in_features) * 0.01)
        
        if bias:
            self.bias = nn.Parameter(torch.zeros(out_features))
        else:
            self.register_parameter('bias', None)

    def forward(self, x):
        # Apply the exponential function to force positivity
        # Positive Weight = exp(internal_weight)
        pos_weight = torch.exp(self.weight)
        
        return F.linear(x, pos_weight, self.bias)

    def extra_repr(self):
        return f'in_features={self.in_features}, out_features={self.out_features}, bias={self.bias is not None}'


class ComplexLinear(torch.nn.Module):
    __constants__ = ["in_features", "out_features"]
    in_features: int
    out_features: int
    weight: torch.Tensor

    def __init__(
        self,
        in_features: int,
        out_features: int,
        bias: bool = True,
        device=None,
        dtype=torch.cfloat,
    ) -> None:
        factory_kwargs = {"device": device, "dtype": dtype}
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weight = torch.nn.Parameter(torch.empty((out_features, in_features), **factory_kwargs))
        if bias:
            self.bias = torch.nn.Parameter(torch.empty(out_features, **factory_kwargs))
        else:
            self.register_parameter("bias", None)
        self.reset_parameters()

    def reset_parameters(self) -> None:
        init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        if self.bias is not None:
            fan_in, _ = init._calculate_fan_in_and_fan_out(self.weight)
            bound = 1 / math.sqrt(fan_in) if fan_in > 0 else 0
            init.uniform_(self.bias, -bound, bound)

    def forward(self, input: torch.Tensor) -> torch.Tensor:
        return F.linear(input, self.weight, self.bias)

    def extra_repr(self) -> str:
        return f"in_features={self.in_features}, out_features={self.out_features}, bias={self.bias is not None}"


class SingledtResHopf(nn.Module):
    def __init__(self, units, min_omega, max_omega, dt, 
                 train_omegas=True, 
                 input_scaler=5.0) -> None:
        super().__init__()

        self.units = units
        self.dt = dt
        self.input_scaler = input_scaler
        
        # Buffers for constants (avoids moving to device manually)
        self.register_buffer('min_omega', torch.tensor(min_omega * 2 * torch.pi))
        self.register_buffer('omega_range', torch.tensor((max_omega - min_omega) * 2 * torch.pi))
        self.register_buffer('mu0', torch.tensor(1.0))
        
        # Use nn.Parameter instead of autograd.Variable
        self.omegas = nn.Parameter(torch.randn(1, units), requires_grad=train_omegas)

    def forward(self, X, r, phi):
        # Pre-calculate trig values once to reuse
        cos_phi = torch.cos(phi)
        sin_phi = torch.sin(phi)
        
        omegas_scaled = torch.sigmoid(self.omegas) * self.omega_range + self.min_omega
        input_r = self.input_scaler * torch.abs(X)
        input_phi = self.input_scaler * X.real * sin_phi
        
        r_new = r + ((self.mu0 - r**2) * r + input_r) * self.dt
        phi_new = phi + (omegas_scaled - input_phi) * self.dt            

        # z = torch.complex(r_new * torch.cos(phi_new), 
        #                   r_new * torch.sin(phi_new))
        z = torch.concat([r_new * torch.cos(phi_new), r_new * torch.sin(phi_new)], dim=1)  # Shape: (batch, units, 2)
        return z, r_new, phi_new

In [ ]:
device = torch.device("cpu")

cues_array = ['fast_0.33', 'fast_0.66', 'slow_0.33', 'slow_0.66']
dt = 0.001
units = 128
input_dim = 18
output_dim = 1

num_steps = 2000
threshold = 1.0

In [ ]:
class BG(nn.Module):

    def __init__(self, input_dim, action_dim, units, d_flag=0):

        super(BG, self).__init__()

        self.input_dim = input_dim
        self.action_dim = action_dim
        self.units = units

        self.d_flag = d_flag

        self.ff = nn.Linear(input_dim, units)

        self.d1 = SingledtResHopf(units, min_omega=min_omega, max_omega=max_omega, 
                                  dt=dt, input_scaler=input_scaler)
        self.d2 = SingledtResHopf(units, min_omega=min_omega, max_omega=max_omega, 
                                  dt=dt, input_scaler=input_scaler)
        self.dp = nn.Linear(2*self.units, action_dim)
        self.ip = nn.Linear(2*self.units, action_dim)

        self.critic = nn.Linear(2*self.units, 1)

        self.gpi_tau = gpi_tau
        self.ip_amp = ip_amp
        self.noise_amp = noise_amp
        self.sigmoid_gain = sigmoid_gain

    def forward(self, state):

        v_prev = torch.zeros(1, 1, device=state.device)
        gpi = torch.zeros(1, self.action_dim, device=state.device)
        r1 = torch.ones(1, self.units, device=state.device)
        r2 = torch.ones(1, self.units, device=state.device)
        phi1 = torch.randn(1, self.units, device=state.device)/10
        phi2 = torch.randn(1, self.units, device=state.device)/10

        value_monitor = torch.zeros(state.shape[0], 1, device=state.device)
        gpi_monitor = torch.zeros(state.shape[0], self.action_dim, device=state.device)
        d1_monitor = torch.zeros(state.shape[0], 2*self.units, device=state.device)
        d2_monitor = torch.zeros(state.shape[0], 2*self.units, device=state.device)
        lD1_monitor = torch.zeros(state.shape[0], 1, device=state.device)
        lD2_monitor = torch.zeros(state.shape[0], 1, device=state.device)

        for t in range(state.shape[0]):

            x = state[t:t+1, :]
            d1out, r1, phi1 = self.d1(x, r1, phi1)
            d2out, r2, phi2 = self.d2(x, r2, phi2)

            value = self.critic(d1out)
            value_monitor[t] = value
            delV = (value - v_prev)
            v_prev = value

            # pathology condition
            if self.d_flag == 1:
                delV = pd_amp * delV

            elif self.d_flag == 2:
                delV = sh_amp * delV
                
            lD1 = torch.sigmoid(self.sigmoid_gain * delV)
            lD2 = torch.sigmoid(-self.sigmoid_gain * delV)

            lD1_monitor[t] = lD1
            lD2_monitor[t] = lD2
            d1_monitor[t] = d1out
            d2_monitor[t] = d2out

            dp_out = torch.relu(self.dp(d1out * lD1))
            ip_out = torch.relu(self.ip(d2out * lD2))
            
            noise = self.noise_amp * torch.randn_like(ip_out) * 2*lD2 - lD2

            gpi = gpi + (dp_out - self.ip_amp*(ip_out + noise)) * self.gpi_tau
            gpi_monitor[t] = gpi

        zipper = {'d1': d1_monitor, 'dp': dp_out,
                  'gpi': gpi, 'value': value_monitor,
                  'd2': d2_monitor, 'ip': ip_out,
                  'lD1': lD1_monitor, 'lD2': lD2_monitor, 'noise': noise
                  }

        return(gpi_monitor, torch.relu(value_monitor).max(), zipper)

In [86]:
import numpy as np, torch, os
from matplotlib import pyplot as plt

for disease_flag in [2]:  # 0: healthy, 1: PD, 2: SH

    model = BG(input_dim, output_dim, units, d_flag=disease_flag)

    print("NOTE: You're currently running model in mode={%d}" % disease_flag)

    optimizer = torch.optim.Adam(model.parameters(), lr=0.002)

    num_epochs = 4000
    pred_times = []
    true_times = []
    mae_monitor = []
    cue_wise_mae = {'fast_0.33': [], 'fast_0.66': [],
                    'slow_0.33': [], 'slow_0.66': []}
    cue_wise_pred = {'fast_0.33': [], 'fast_0.66': [],
                    'slow_0.33': [], 'slow_0.66': []}

    for episode in range(num_epochs):
        if not episode:
            print("You're running in disease mode:", disease_flag)
        state, target, speed, cutoof = generate_continious_occlusion_data(input_dim, num_steps)
        state = np.expand_dims(state, 1)
        state = torch.tensor(state, dtype=torch.float32, device=device)
        (gpi, value, _) = model(state)

        error = torch.abs(gpi[target] - threshold)
        reward = torch.exp(-error)
        
        td_error = reward - value

        if model.d_flag == 1:
            # td_error = torch.clamp(td_error, max=0.3)
            td_error = 0.5*td_error
        elif model.d_flag == 2:
            # td_error = torch.clamp(td_error, min=1.2)
            td_error = 1.7*td_error

        loss = td_error**2 - value #+ error*5

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Find first crossing of threshold
        crossed = (gpi >= threshold).nonzero(as_tuple=True)[0]
        pred_t = crossed[0].item() if len(crossed) > 0 else (num_steps - 1)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)   # tighter clip
        mae = target - pred_t
        mae_monitor.append(mae)
        
        if (episode+1) % 500 == 0:
            print(f"Epoch {episode+1:4d} | Loss {loss.item():.4f} | MAE {mae:4.1f} | Pred/Target {pred_t:d}/{target:d}")
            plt.plot(mae_monitor, label='MAE over time')
            plt.show()
            clear_output(wait=True)
            
        if (episode + 1) % 500 == 0:
            torch.save(model.state_dict(), f"new_amt_model{disease_flag}.pt")

In [ ]:
# num_iter = 100
# d_flag = 2
# print(f"testing for d_flag : {d_flag}")
# pred_times = []
# true_times = []
# cue_wise_mae = {'fast_0.33': [], 'fast_0.66': [],
#                 'slow_0.33': [], 'slow_0.66': []}

# for cue_tag in ['fast_33', 'fast_66', 'slow_33', 'slow_66']:

#     for iter in range(num_iter):

#         model = BG(input_dim, output_dim, units, d_flag=d_flag)
#         model.load_state_dict(torch.load(f"new_amt_model{d_flag}.pt"))
#         state, target, speed, cutoff = generate_occlusion_data(input_dim, num_steps, force_trial=True, ttype=cue_tag)
#         # print(cue_tag, speed, cutoff)
#         state = torch.tensor(state, dtype=torch.float32, device=device).reshape(-1, 1, input_dim)
#         (gpi, value, _) = model(state)

#         # Find first crossing of threshold
#         crossed = (gpi >= threshold).nonzero(as_tuple=True)[0]
#         pred_t = crossed[0].item() if len(crossed) > 0 else (num_steps - 1)

#         mae = target - pred_t
#         cue_key = f"{speed}_{cutoff}"
#         cue_wise_mae[cue_key].append(mae)

# mean_bars = [np.mean(cue_wise_mae[cue]) for cue in cues_array]
# std_errors = [np.std(cue_wise_mae[cue])/np.sqrt(len(cue_wise_mae[cue])) for cue in cues_array]
# plt.bar(cues_array, mean_bars, yerr=std_errors)
# plt.show()

In [87]:
num_iter = 100
master_pred_monitor = [[] for _ in range(3)]
master_target_monitor = [[] for _ in range(3)]
master_mae_monitor = [[] for _ in range(3)]

for d_flag in [0, 1, 2]:

    print(f"D_flag : {d_flag}")
    pred_times = []
    true_times = []
    
    cue_wise_mae = {'fast_0.33': [], 'fast_0.66': [],
                    'slow_0.33': [], 'slow_0.66': []}
    cue_wise_pred = {'fast_0.33': [], 'fast_0.66': [],
                    'slow_0.33': [], 'slow_0.66': []}
    
    for cue_tag in ['fast_33', 'fast_66', 'slow_33', 'slow_66']:

        for iter in range(num_iter):

            model = BG(input_dim, output_dim, units, d_flag=d_flag)
            model.load_state_dict(torch.load(f"new_amt_model{d_flag}.pt"))
            state, target, speed, cutoff = generate_occlusion_data(input_dim, num_steps, force_trial=True, ttype=cue_tag)
            # print(cue_tag, speed, cutoff)
            state = torch.tensor(state, dtype=torch.float32, device=device).reshape(-1, 1, input_dim)
            (gpi, value, _) = model(state)

            # Find first crossing of threshold
            crossed = (gpi >= threshold).nonzero(as_tuple=True)[0]
            pred_t = crossed[0].item() if len(crossed) > 0 else (num_steps - 1)

            mae = target - pred_t
            cue_key = f"{speed}_{cutoff}"
            cue_wise_mae[cue_key].append(mae)
            cue_wise_pred[cue_key].append(pred_t)

    master_mae_monitor[d_flag] = cue_wise_mae
    master_pred_monitor[d_flag] = cue_wise_pred

In [4]:
master_mae_monitor = [np.load('nc_mae.npy', allow_pickle=True), 
                      np.load('pd_mae.npy', allow_pickle=True), 
                      np.load('sh_mae.npy', allow_pickle=True)]

In [13]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import sem # For Standard Error of the Mean

plt.rcParams.update({'font.size': 14})
groups = ['fast-long', 'fast-short', 'slow-long', 'slow-short']
means_d0, errs_d0 = [], []
means_d1, errs_d1 = [], []
means_d2, errs_d2 = [], []

# Load data
# nc_master = master_mae_monitor[0]
# pd_master = master_mae_monitor[1]
# sh_master = master_mae_monitor[2]

nc_master, pd_master, sh_master = master_mae_monitor[0].item(), master_mae_monitor[1].item(), master_mae_monitor[2].item()

time_scalr = 15

# Extract means and errors for each group
for g in ['fast_0.33', 'fast_0.66', 'slow_0.33', 'slow_0.66']:
    # d_flag = 0 (NC)
    data_d0 = np.array(nc_master[g]) * time_scalr
    means_d0.append(np.mean(data_d0))
    errs_d0.append(np.std(data_d0)) # Standard Error
    
    # d_flag = 1 (PD)
    data_d1 = np.array(pd_master[g]) * time_scalr
    means_d1.append(np.mean(data_d1))
    errs_d1.append(np.std(data_d1))
    
    # d_flag = 2 (SH)
    data_d2 = np.array(sh_master[g]) * time_scalr
    means_d2.append(np.mean(data_d2))
    errs_d2.append(np.std(data_d2))

# Plotting
x = np.arange(len(groups))  # label locations
width = 0.25  # width of the bars

fig, ax = plt.subplots(figsize=(10, 6))

# Positioning the bars side-by-side with error bars
# yerr provides the error values, capsize adds the horizontal lines at the tips
rects1 = ax.bar(x - width, np.array(means_d0), width, yerr=np.array(errs_d0), label='NC', capsize=5, color='green')
rects2 = ax.bar(x, np.array(means_d1), width, yerr=np.array(errs_d1), label='PD', capsize=5, color='orange')
rects3 = ax.bar(x + width, np.array(means_d2), width, yerr=np.array(errs_d2), label='SCZ', capsize=5, color='blue', alpha=0.7)

# Add labels, title and custom x-axis tick labels
ax.set_ylabel('Mean error (in ms)')
ax.set_title('Difference between objective and predicted time')
ax.set_xticks(x)
ax.set_xticklabels(groups)
ax.legend(loc='lower left')

plt.tight_layout()
plt.savefig("final_amt_bg.png", dpi=400)

In [12]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import ptitprince as pt
import numpy as np
from scipy.stats import ttest_ind

# 1. Prepare Data
# plt.rcParams.update({'font.size': 14})
data_list = []
keys = ['slow_0.33', 'slow_0.66']
groups_labels = ['slow-long', 'slow-short']
time_scalr = 15
nc_master, pd_master, sh_master = master_mae_monitor[0].item(), master_mae_monitor[1].item(), master_mae_monitor[2].item()

for i, key in enumerate(keys):
    for label, master_dict in zip(['NC', 'PD', 'SCZ'], [nc_master, pd_master, sh_master]):
        values = np.array(master_dict[key]) * time_scalr
        for v in values:
            data_list.append({'Type': groups_labels[i], 'Error': v, 'Group': label})

df = pd.DataFrame(data_list)

# 2. Plotting
plt.rcParams.update({'font.size': 14})
plt.figure(figsize=(10, 6))

palette = {'NC': 'green', 'PD': 'orange', 'SCZ': 'blue'}

# Setting move=0.0 aligns points and boxes
ax = pt.RainCloud(
    data=df, x='Type', y='Error', hue='Group', 
    palette=palette, orient='v', 
    width_viol=0.5, width_box=0.15,
    alpha=0.8, dodge=True, move=0.0, 
    point_size=2, jitter=0.05, box_showfliers=False
)

# 3. Precise Statistical Alignment
def get_sig_label(p):
    if p < 0.001: return '***'
    if p < 0.01: return '**'
    if p < 0.05: return '*'
    return None

# These offsets align exactly with the center of the dodged box plots
x_offsets = [-0.1, 0, 0.1] 
group_order = ['NC', 'PD', 'SCZ']
pairs = [('NC', 'PD'), ('NC', 'SCZ'), ('PD', 'SCZ')]

y_max = df['Error'].max()
y_min = df['Error'].min()
# Calculate a dynamic step for the bars based on data range
y_range = y_max - y_min
line_step = y_range * 0.08 

for i, cond in enumerate(groups_labels):
    cond_data = df[df['Type'] == cond]
    # Start the first bar just above the highest point/cloud
    current_y = cond_data['Error'].max() + (y_range * 0.05)
    
    for (g1, g2) in pairs:
        v1 = cond_data[cond_data['Group'] == g1]['Error']
        v2 = cond_data[cond_data['Group'] == g2]['Error']
        
        stat, p = ttest_ind(v1, v2, nan_policy='omit', equal_var=False)
        sig_text = get_sig_label(p)
        
        if sig_text:
            # i is the category index (0 for slow-long, 1 for slow-short)
            x1 = i + x_offsets[group_order.index(g1)]
            x2 = i + x_offsets[group_order.index(g2)]
            
            # Draw the bracket
            # The vertical tips of the bracket help visually ground the line to the box
            ax.plot([x1, x1, x2, x2], [current_y, current_y + (line_step*0.2), current_y + (line_step*0.2), current_y], 
                    lw=1.5, c='black')
            
            # Place the asterisk
            ax.text((x1 + x2) * 0.5, current_y + (line_step*0.2), sig_text, 
                    ha='center', va='bottom', fontsize=12, fontweight='bold')
            
            # Bump the height for the next comparison
            current_y += line_step

# 4. Final Polish
plt.title('Error distributions for Slow Cues')
plt.ylabel('error (in ms)')
plt.ylim(y_min - 500, current_y + 500)

handles, labels = ax.get_legend_handles_labels()
plt.legend(handles[0:3], labels[0:3], title='', loc='lower right', frameon=True)

sns.despine()
plt.tight_layout()
plt.savefig("final_amt_bg_slow.png", dpi=400)

In [ ]:
# np.save('nc_mae.npy', master_mae_monitor[0])
# np.save('pd_mae.npy', master_mae_monitor[1])
# np.save('sh_mae.npy', master_mae_monitor[2])

In [ ]:
master_mae_monitor[0] = np.load('nc_mae.npy', allow_pickle=True)
master_mae_monitor[1] = np.load('pd_mae.npy', allow_pickle=True)
master_mae_monitor[2] = np.load('sh_mae.npy', allow_pickle=True)